# Demo: delegation flow of the multi-agent orchestrator

Run this notebook from the project root kernel (`.venv`). It needs `GOOGLE_API_KEY` in `.env`.

Expected flow: **supervisor -> researcher -> supervisor -> analyst -> supervisor -> synthesizer**.

In [ ]:
import os, sys
os.chdir(os.path.abspath('..')) if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, os.getcwd())
from src.graph import build_graph, initial_state
from src.main import DEFAULT_TASK, format_event
graph = build_graph()

## Graph topology

In [ ]:
from IPython.display import Markdown
Markdown('```mermaid\n' + graph.get_graph().draw_mermaid() + '\n```')

## Run and watch each delegation step

In [ ]:
contributions, decisions = [], []
for event in graph.stream(initial_state(DEFAULT_TASK), config={'recursion_limit': 30}, stream_mode='updates'):
    for node, update in event.items():
        print(format_event(node, update), '\n')
        contributions += update.get('contributions', [])
        if node == 'supervisor':
            decisions.append(update['next_agent'])

## Who contributed what (shared state)

In [ ]:
for c in contributions:
    print(f"decision {c['step']} | {c['agent']:<10} | {len(c['output'])} chars")
print('supervisor route:', ' -> '.join(decisions))